# Phase 10b: Seed-ensemble evaluation (post-hoc, no training, CPU only)
Averages the **predicted probabilities** of the seeds already evaluated in Phase 10 (`results/phase10/<model>/seed<k>/test_predictions.csv`) and scores the ensemble on the test set.
**Rules fixed in advance (so nothing is chosen on test data):** (1) the ensemble contains ALL seeds of a model, (2) both models are ensembled the same way, (3) the proposed model is the lambda selected on VALIDATION in Phase 9. Other lambdas, the ensemble-size curve and the baseline+proposed pool are reported as information only and must not be used to pick a model.
Needs only Phase 10 outputs plus `data/test_sent_emo.csv` (for dialogue ids). Does not change any existing file; writes to `results/ensemble/`.

In [ ]:
# Required Imports
import os, sys, json, itertools
import numpy as np
import pandas as pd
print("Libraries imported successfully.")

In [ ]:
# 1. Configuration and Phase 10 products
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
config = json.load(open(os.path.join(BASE_DIR, "config.json")))
RESULTS_DIR = config["paths"]["results_dir"]
EMOTIONS = config["emotions"]; C = len(EMOTIONS)
P_COLS = ["p_" + e for e in EMOTIONS]
selected = json.load(open(os.path.join(RESULTS_DIR, "phase9_selected_lambda.json")))
BASELINE, PROPOSED = "baseline", selected["selected_experiment"]
P10_DIR = os.path.join(RESULTS_DIR, "phase10")
OUT_DIR = os.path.join(RESULTS_DIR, "ensemble"); os.makedirs(OUT_DIR, exist_ok=True)
N_BOOT = int(os.environ.get("MELD_ENS_BOOT", 2000))
test_df = pd.read_csv(os.path.join(BASE_DIR, "data", "test_sent_emo.csv"), keep_default_na=False)
DLG = test_df["Dialogue_ID"].values
models = sorted(d for d in os.listdir(P10_DIR) if os.path.isdir(os.path.join(P10_DIR, d)) and (d == "baseline" or d.startswith("proposed_lam")))
assert BASELINE in models and PROPOSED in models, f"Phase 10 folders found: {models}"

def load_model(name):
    seeds = sorted(int(d[4:]) for d in os.listdir(os.path.join(P10_DIR, name)) if d.startswith("seed") and os.path.exists(os.path.join(P10_DIR, name, d, "test_predictions.csv")))
    dfs = {s: pd.read_csv(os.path.join(P10_DIR, name, f"seed{s}", "test_predictions.csv")) for s in seeds}
    return seeds, dfs

DATA = {m: load_model(m) for m in models}
Y = DATA[BASELINE][1][DATA[BASELINE][0][0]]["y_true"].values
N = len(Y)
assert N == len(test_df), f"{N} predictions vs {len(test_df)} test rows"
for m, (seeds, dfs) in DATA.items():                       # every file must describe the SAME test rows in the SAME order
    for s, d in dfs.items():
        assert (d["row_idx"].values == np.arange(N)).all() and (d["y_true"].values == Y).all(), f"{m} seed{s}: rows/labels differ from baseline"
        assert np.allclose(d[P_COLS].values.sum(1), 1.0, atol=1e-3), f"{m} seed{s}: probabilities do not sum to 1"
print(f"Models: {models}\nBaseline='{BASELINE}' | proposed='{PROPOSED}' (lambda selected on VALIDATION) | test utterances: {N} | dialogues: {len(np.unique(DLG))}")
print({m: seeds for m, (seeds, _) in DATA.items()})

In [ ]:
# 2. Metrics (same definitions as Phase 10: macro-F1 from the confusion matrix; classes absent from both truth and prediction score 0)
def conf_flat(y, p): return np.bincount(np.asarray(y) * C + np.asarray(p), minlength=C * C).astype(np.float64)
def f1_pc(conf_f):
    conf = np.asarray(conf_f).reshape(C, C); tp, row, col = np.diag(conf), conf.sum(1), conf.sum(0)
    return np.where(row + col > 0, 2 * tp / np.maximum(row + col, 1e-300), 0.0)
def metrics(y, p):
    f = f1_pc(conf_flat(y, p)); support = np.bincount(y, minlength=C)
    return {"accuracy": float((y == p).mean()), "macro_f1": float(f.mean()), "weighted_f1": float((f * support).sum() / support.sum()), **{f"f1_{e}": float(v) for e, v in zip(EMOTIONS, f)}}
def ens_probs(model, seeds=None):
    sd, dfs = DATA[model]; use = sd if seeds is None else list(seeds)
    return np.mean([dfs[s][P_COLS].values for s in use], axis=0)
def dialogue_bootstrap(y, p_new, p_ref, ids, n_boot, seed=0, alpha=0.05):
    ids_u, inv = np.unique(ids, return_inverse=True); D = len(ids_u)
    cn, cr = np.zeros((D, C * C)), np.zeros((D, C * C))
    np.add.at(cn, (inv, y * C + p_new), 1); np.add.at(cr, (inv, y * C + p_ref), 1)
    rng = np.random.RandomState(seed); diffs = np.empty(n_boot)
    for b in range(n_boot):
        cnt = np.bincount(rng.randint(0, D, D), minlength=D).astype(np.float64)
        diffs[b] = f1_pc(cnt @ cn).mean() - f1_pc(cnt @ cr).mean()
    point = f1_pc(cn.sum(0)).mean() - f1_pc(cr.sum(0)).mean()
    lo, hi = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {"point": float(point), "ci_low": float(lo), "ci_high": float(hi), "n_boot": n_boot}
print("Helpers defined.")

In [ ]:
# 3. Single seeds vs seed-ensemble (all seeds, probability average)
rows = []
for m in models:
    seeds, dfs = DATA[m]
    single = pd.DataFrame([metrics(Y, dfs[s][P_COLS].values.argmax(1)) for s in seeds])
    ens = metrics(Y, ens_probs(m).argmax(1))
    role = "baseline" if m == BASELINE else ("PROPOSED (selected on validation)" if m == PROPOSED else "info only")
    rows.append({"model": m, "role": role, "n_seeds": len(seeds),
                 "single_acc_mean": single["accuracy"].mean(), "single_acc_std": single["accuracy"].std(ddof=1) if len(seeds) > 1 else np.nan,
                 "single_macroF1_mean": single["macro_f1"].mean(), "single_macroF1_std": single["macro_f1"].std(ddof=1) if len(seeds) > 1 else np.nan,
                 "ens_acc": ens["accuracy"], "ens_macroF1": ens["macro_f1"], "ens_weightedF1": ens["weighted_f1"],
                 "ens_gain_macroF1": ens["macro_f1"] - single["macro_f1"].mean(), "ens_gain_acc": ens["accuracy"] - single["accuracy"].mean()})
summary = pd.DataFrame(rows)
pd.set_option("display.width", 250)
print(summary.round(4).to_string(index=False))
summary.to_csv(os.path.join(OUT_DIR, "ensemble_summary.csv"), index=False)

pc = pd.DataFrame({m: {e: metrics(Y, ens_probs(m).argmax(1))[f"f1_{e}"] for e in EMOTIONS} for m in (BASELINE, PROPOSED)})
pc["diff (proposed - baseline)"] = pc[PROPOSED] - pc[BASELINE]
print("\nPer-class F1 of the seed ensembles:"); print(pc.round(4).to_string())
pc.to_csv(os.path.join(OUT_DIR, "ensemble_per_class.csv"))

In [ ]:
# 4. Ensemble vs ensemble: paired dialogue-level bootstrap (resamples test dialogues), the headline comparison
yb, yp = ens_probs(BASELINE).argmax(1), ens_probs(PROPOSED).argmax(1)
boot = dialogue_bootstrap(Y, yp, yb, DLG, N_BOOT, seed=0)
verdict = ("proposed ensemble higher" if boot["ci_low"] > 0 else "baseline ensemble higher" if boot["ci_high"] < 0 else "no significant difference (the interval includes 0)")
print(f"Macro-F1, seed ensembles: baseline {metrics(Y, yb)['macro_f1']:.4f} | proposed {metrics(Y, yp)['macro_f1']:.4f}")
print(f"Paired dialogue bootstrap (proposed - baseline): {boot['point']:+.4f}, 95% CI [{boot['ci_low']:+.4f}, {boot['ci_high']:+.4f}] -> {verdict}")
print("Note: this CI reflects test-set sampling only; seed variance is removed by the averaging, so it is NOT comparable with the seed-level paired t-test in Phase 10.")

# Information only: ensemble-size curve (mean over all subsets of k seeds) and the baseline + proposed pool
curve = []
for m in (BASELINE, PROPOSED):
    seeds = DATA[m][0]
    for k in range(1, len(seeds) + 1):
        vals = [metrics(Y, ens_probs(m, sub).argmax(1)) for sub in itertools.combinations(seeds, k)]
        curve.append({"model": m, "k_seeds": k, "n_subsets": len(vals), "macro_f1": np.mean([v["macro_f1"] for v in vals]), "accuracy": np.mean([v["accuracy"] for v in vals])})
curve = pd.DataFrame(curve)
print("\nEnsemble-size curve (information only):"); print(curve.round(4).to_string(index=False))
curve.to_csv(os.path.join(OUT_DIR, "ensemble_size_curve.csv"), index=False)
pool = np.mean([ens_probs(BASELINE), ens_probs(PROPOSED)], axis=0).argmax(1)
print(f"\nPool baseline+proposed ({len(DATA[BASELINE][0]) + len(DATA[PROPOSED][0])} runs, information only): acc {metrics(Y, pool)['accuracy']:.4f}, macro-F1 {metrics(Y, pool)['macro_f1']:.4f}")

In [ ]:
# 4b. Validation-weighted seed ensemble (STEP L). Rule fixed in advance and identical for both models: seed weights = softmax(validation score / T), T = 0.01,
# using the per-seed VALIDATION scores of Phase 9 (macro-F1, or accuracy when MELD_SELECT_METRIC=accuracy). Reported NEXT TO the uniform ensemble above; the headline stays the uniform ensemble.
METRIC_SEL = os.environ.get("MELD_SELECT_METRIC", "macro_f1").strip().lower()
VAL_FILE = os.path.join(RESULTS_DIR, "phase9_val_per_seed_accuracy.csv" if METRIC_SEL == "accuracy" else "phase9_val_per_seed.csv")
TEMP = float(os.environ.get("MELD_ENS_WEIGHT_TEMP", 0.01))
def softmax_weights(v, temp):
    v = np.asarray(v, dtype=np.float64); e = np.exp((v - v.max()) / temp); return e / e.sum()
WEIGHTED_RESULT = {"status": "skipped", "reason": f"{VAL_FILE} not found"}
if os.path.exists(VAL_FILE):
    _ps = pd.read_csv(VAL_FILE, index_col=0)
    def wens_probs(model):
        sd, dfs = DATA[model]; w = softmax_weights([_ps.loc[s, model] for s in sd], TEMP)
        return np.tensordot(w, np.stack([dfs[s][P_COLS].values for s in sd]), axes=1), w
    wb, wbw = wens_probs(BASELINE); wp, wpw = wens_probs(PROPOSED)
    mb_u, mp_u = metrics(Y, ens_probs(BASELINE).argmax(1)), metrics(Y, ens_probs(PROPOSED).argmax(1))
    mb_w, mp_w = metrics(Y, wb.argmax(1)), metrics(Y, wp.argmax(1))
    bootw = dialogue_bootstrap(Y, wp.argmax(1), wb.argmax(1), DLG, N_BOOT, seed=0)
    tab = pd.DataFrame({"baseline_uniform": [mb_u["accuracy"], mb_u["macro_f1"]], "proposed_uniform": [mp_u["accuracy"], mp_u["macro_f1"]],
                        "baseline_weighted": [mb_w["accuracy"], mb_w["macro_f1"]], "proposed_weighted": [mp_w["accuracy"], mp_w["macro_f1"]]}, index=["accuracy", "macro_f1"])
    print(f"Validation-weighted seed ensembles (score = validation {METRIC_SEL}, T = {TEMP}); weights baseline {np.round(wbw, 3).tolist()}, proposed {np.round(wpw, 3).tolist()}")
    print(tab.round(4).to_string())
    print(f"Paired dialogue bootstrap, weighted ensembles (proposed - baseline, macro-F1): {bootw['point']:+.4f}, 95% CI [{bootw['ci_low']:+.4f}, {bootw['ci_high']:+.4f}]")
    tab.to_csv(os.path.join(OUT_DIR, "ensemble_weighted.csv"))
    assert abs(wbw.sum() - 1) < 1e-9 and abs(wpw.sum() - 1) < 1e-9 and (wbw > 0).all() and (wpw > 0).all(), "weights must be a probability vector"
    assert np.allclose(softmax_weights([0.5, 0.5, 0.5], TEMP), 1 / 3), "equal validation scores must give the plain average"
    WEIGHTED_RESULT = {"status": "ok", "metric": METRIC_SEL, "temperature": TEMP, "weights_baseline": wbw.tolist(), "weights_proposed": wpw.tolist(),
                       "baseline": {"accuracy": mb_w["accuracy"], "macro_f1": mb_w["macro_f1"]}, "proposed": {"accuracy": mp_w["accuracy"], "macro_f1": mp_w["macro_f1"]},
                       "paired_dialogue_bootstrap_macro_f1": bootw}
else:
    print("Validation-weighted ensemble skipped:", WEIGHTED_RESULT["reason"])


In [ ]:
# 5. Verification (independent re-computation; a failure stops the notebook)
from sklearn.metrics import f1_score, accuracy_score
res = []
def check(name, cond, detail=""):
    res.append({"test": name, "passed": bool(cond), "detail": detail}); print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))
for m in (BASELINE, PROPOSED):
    p = ens_probs(m); pr = p.argmax(1); mm = metrics(Y, pr)
    check(f"E1 [{m}] ensemble macro-F1 / accuracy equal sklearn", abs(mm["macro_f1"] - f1_score(Y, pr, average="macro", labels=range(C), zero_division=0)) < 1e-9 and abs(mm["accuracy"] - accuracy_score(Y, pr)) < 1e-12)
    check(f"E2 [{m}] ensemble probabilities are valid distributions", np.allclose(p.sum(1), 1.0, atol=1e-3) and (p >= 0).all())
    s0 = DATA[m][0][0]
    check(f"E3 [{m}] a one-seed 'ensemble' equals that seed's own prediction", (ens_probs(m, [s0]).argmax(1) == DATA[m][1][s0]["y_pred"].values).all())
    perm = np.random.RandomState(1).permutation(len(DATA[m][0]))
    check(f"E4 [{m}] ensemble does not depend on seed order", np.allclose(ens_probs(m, [DATA[m][0][i] for i in perm]), p))
same = dialogue_bootstrap(Y, yb, yb, DLG, 200, seed=0); perfect = dialogue_bootstrap(Y, Y, yb, DLG, 200, seed=0)
check("E5 bootstrap: identical predictions give difference 0 and CI [0, 0]", same["point"] == 0 and same["ci_low"] == 0 and same["ci_high"] == 0)
check("E6 bootstrap: perfect predictions beat the baseline ensemble with a CI above 0", perfect["point"] > 0 and perfect["ci_low"] > 0)
check("E7 ensemble-size curve is anchored: k=n equals the full ensemble", abs(float(curve[(curve.model == PROPOSED) & (curve.k_seeds == len(DATA[PROPOSED][0]))]["macro_f1"].iloc[0]) - metrics(Y, yp)["macro_f1"]) < 1e-12)
check("E8 no test-based selection: the proposed model is the validation-selected lambda", PROPOSED == selected["selected_experiment"])
failed = [r for r in res if not r["passed"]]
json.dump({"n_tests": len(res), "n_failed": len(failed), "results": res}, open(os.path.join(OUT_DIR, "ensemble_verification.json"), "w"), indent=2)
assert not failed, f"{len(failed)} check(s) failed: {[r['test'] for r in failed]}"
print(f"\nAll {len(res)} ensemble checks passed.")

In [ ]:
# 6. Save the result
b_s, p_s = summary[summary.model == BASELINE].iloc[0], summary[summary.model == PROPOSED].iloc[0]
out = {"rule": "all seeds, probability average, same for both models; proposed = lambda selected on validation; nothing chosen on test data",
       "baseline": {"model": BASELINE, "n_seeds": int(b_s.n_seeds), "single_macro_f1_mean": float(b_s.single_macroF1_mean), "single_macro_f1_std": float(b_s.single_macroF1_std), "ensemble_macro_f1": float(b_s.ens_macroF1), "ensemble_accuracy": float(b_s.ens_acc)},
       "proposed": {"model": PROPOSED, "n_seeds": int(p_s.n_seeds), "single_macro_f1_mean": float(p_s.single_macroF1_mean), "single_macro_f1_std": float(p_s.single_macroF1_std), "ensemble_macro_f1": float(p_s.ens_macroF1), "ensemble_accuracy": float(p_s.ens_acc)},
       "paired_dialogue_bootstrap_proposed_minus_baseline": {**boot, "verdict": verdict}, "n_test_utterances": int(N), "n_test_dialogues": int(len(np.unique(DLG)))}
out["validation_weighted_ensemble"] = WEIGHTED_RESULT          # information only; the headline is the uniform ensemble
json.dump(out, open(os.path.join(OUT_DIR, "ensemble_results.json"), "w"), indent=2)
print(json.dumps(out, indent=2))
print("\nSaved to", OUT_DIR, "(ensemble_results.json, ensemble_summary.csv, ensemble_per_class.csv, ensemble_size_curve.csv, ensemble_verification.json)")